# Lista 2 - geometria computacional

Implementei as primitivas e os algoritmos desta lista em C++, expondo-os ao Python pela API pública `tca.geometry`. O notebook concentra a formulação matemática, os casos de verificação e as visualizações interativas; o código Python presente aqui é usado apenas para preparar os experimentos e apresentar os resultados.

As figuras são geradas com Plotly e salvas em HTML em `assignments/02-geometry/artifacts/`.

### Sobre a implementação

As implementações seguem as convenções adotadas no projeto:

- plano $\mathbb{R}^2$;
- aritmética em `double`;
- orientação anti-horária positiva;
- tolerância numérica centralizada no kernel C++;
- classificação `0 = exterior`, `1 = interior` e `2 = fronteira`, quando aplicável.

A célula abaixo concentra os imports e as pequenas rotinas de visualização reutilizadas ao longo da lista. Os algoritmos geométricos permanecem no kernel C++.

In [1]:
from copy import deepcopy
from math import ceil
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from tca.algorithms.sorting import sort
from tca.geometry import (
    barycentric_coordinates,
    entre,
    first_horizontal_intersection,
    is_convex,
    locate_from_first_intersection,
    locate_point_in_triangle,
    oriented_area,
    pseudo_angle,
    triangles_are_disjoint,
)


# Diretório do projeto e pasta de artefatos da Lista 2.
ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").exists()
)

ARTIFACTS_DIR = ROOT / "assignments" / "02-geometry" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)


LOCATION_LABELS = {
    0: "exterior",
    1: "interior",
    2: "fronteira",
}

LOCATION_COLORS = {
    0: "crimson",
    1: "seagreen",
    2: "darkorange",
}


# Repete o primeiro vértice para fechar uma linha poligonal no Plotly.
def close_path(vertices):
    vertices = np.asarray(vertices, dtype=np.float64)
    return np.vstack((vertices, vertices[0]))


# Figura geométrica padrão usada nas visualizações da lista.
def make_figure(*, x_title="x", y_title="y", equal_axes=True):
    fig = go.Figure()

    fig.update_xaxes(title_text=x_title, zeroline=True)
    fig.update_yaxes(title_text=y_title, zeroline=True)

    if equal_axes:
        fig.update_xaxes(scaleanchor="y", scaleratio=1)

    return fig


# Adiciona um polígono a uma figura já criada.
def add_polygon(
    fig,
    vertices,
    *,
    name="Polígono",
    fill=False,
    vertex_labels=False,
    line_color=None,
    fillcolor=None,
    showlegend=True,
):
    closed = close_path(vertices)

    line = {"width": 2}
    marker = {"size": 8}

    if line_color is not None:
        line["color"] = line_color
        marker["color"] = line_color

    mode = "lines+markers+text" if vertex_labels else "lines+markers"
    text = (
        [*(f"P{i + 1}" for i in range(len(vertices))), ""]
        if vertex_labels
        else None
    )

    fig.add_trace(
        go.Scatter(
            x=closed[:, 0],
            y=closed[:, 1],
            mode=mode,
            text=text,
            textposition="top center",
            fill="toself" if fill else None,
            fillcolor=fillcolor,
            line=line,
            marker=marker,
            name=name,
            showlegend=showlegend,
        )
    )

    return fig


# Compõe figuras simples em uma única grade de subplots.
def compose_figures(
    figures,
    *,
    titles=None,
    rows=1,
    cols=None,
    title=None,
    width=1200,
    height=600,
    equal_axes=True,
    horizontal_spacing=0.08,
    vertical_spacing=0.12,
    showlegend=True,
):
    if cols is None:
        cols = ceil(len(figures) / rows)

    titles = titles or [""] * len(figures)
    equal_flags = (
        [equal_axes] * len(figures)
        if isinstance(equal_axes, bool)
        else list(equal_axes)
    )

    fig = make_subplots(
        rows=rows,
        cols=cols,
        subplot_titles=titles,
        horizontal_spacing=horizontal_spacing,
        vertical_spacing=vertical_spacing,
    )

    for index, source in enumerate(figures):
        row = index // cols + 1
        col = index % cols + 1

        # Cada painel é construído como uma figura comum e depois copiado.
        for trace in source.data:
            fig.add_trace(deepcopy(trace), row=row, col=col)

        x_title = getattr(source.layout.xaxis.title, "text", None)
        y_title = getattr(source.layout.yaxis.title, "text", None)

        fig.update_xaxes(
            title_text=x_title,
            zeroline=True,
            row=row,
            col=col,
        )
        fig.update_yaxes(
            title_text=y_title,
            zeroline=True,
            row=row,
            col=col,
        )

        if equal_flags[index]:
            axis_index = index + 1
            yaxis = "y" if axis_index == 1 else f"y{axis_index}"
            fig.update_xaxes(
                scaleanchor=yaxis,
                scaleratio=1,
                row=row,
                col=col,
            )

    fig.update_layout(
        title=title,
        width=width,
        height=height,
        showlegend=showlegend,
    )

    return fig


# Salva a figura interativa como HTML e retorna a própria figura.
def save_figure(fig, filename):
    output_path = ARTIFACTS_DIR / filename
    fig.write_html(output_path, include_plotlyjs="cdn", full_html=True)
    print(f"Figura salva em: {output_path.relative_to(ROOT)}")
    return fig

## Questão 1 — Pseudo-ângulo

A primitiva `pseudo_angle` associa a direção de um vetor não nulo a um valor no intervalo $[0,8)$, preservando a ordem angular sem utilizar funções trigonométricas. O mapeamento inicia no semieixo positivo de $x$ e cresce no sentido anti-horário.

Para um vetor não nulo, o cálculo utiliza:

- **3 comparações** para determinar o quadrante e o trecho correspondente;
- **1 divisão**;
- **no máximo 1 soma ou subtração**;
- **até 2 inversões de sinal**;
- **0 multiplicações**;
- **0 raízes quadradas**;
- **0 operações de valor absoluto**;
- **0 funções trigonométricas**.

A implementação inclui ainda uma verificação adicional do vetor nulo por meio de `is_zero`. Essa validação não é exigida pelo cálculo solicitado no enunciado e, portanto, não entra na contagem acima. Ela foi adotada como decisão de projeto para rejeitar explicitamente uma entrada sem direção definida e manter a mesma política de tolerância numérica das demais primitivas.

Como `is_zero` utiliza `std::abs` internamente, essa etapa adicional pode executar até dois testes de nulidade e duas operações de valor absoluto. Esse custo pertence à validação da entrada, e não ao cálculo do pseudo-ângulo propriamente dito.

In [2]:
# Valores de referência nos quatro semieixos.
axis_vectors = {
    "+x": np.array([1.0, 0.0], dtype=np.float64),
    "+y": np.array([0.0, 1.0], dtype=np.float64),
    "-x": np.array([-1.0, 0.0], dtype=np.float64),
    "-y": np.array([0.0, -1.0], dtype=np.float64),
}

for name, vector in axis_vectors.items():
    print(f"{name}: pseudo_angle = {pseudo_angle(vector):.1f}")

# O vetor nulo e um vetor dentro da tolerância são rejeitados.
for vector in (
    np.array([0.0, 0.0], dtype=np.float64),
    np.array([1e-13, 1e-13], dtype=np.float64),
):
    try:
        pseudo_angle(vector)
    except ValueError as error:
        print(f"Entrada rejeitada {vector}: {error}")

+x: pseudo_angle = 0.0
+y: pseudo_angle = 2.0
-x: pseudo_angle = 4.0
-y: pseudo_angle = 6.0
Entrada rejeitada [0. 0.]: pseudo_angle: vector cannot be the zero vector
Entrada rejeitada [1.e-13 1.e-13]: pseudo_angle: vector cannot be the zero vector


### Verificação da ordenação angular

Para exercitar a primitiva, utilizo os pontos do produto cartesiano

$$
\{-3,-2,-1,0,1,2,3\}^2,
$$

excluindo $(0,0)$. Pequenas perturbações de até $10^{-3}$ são adicionadas aos pontos sobre eixos e diagonais para exercitar as regiões de transição das comparações internas.

A geração utiliza `SEED = 42`. No último painel, os pseudo-ângulos são ordenados pelo Merge Sort em C++ já implementado no projeto.

In [3]:
SEED = 42
SMALL_ERROR = 1e-3

rng = np.random.default_rng(SEED)
values = [-3, -2, -1, 0, 1, 2, 3]

q1_points = np.array(
    [(x, y) for x in values for y in values if not (x == 0 and y == 0)],
    dtype=np.float64,
)

# Perturba apenas pontos situados nas fronteiras usadas nas comparações.
for point in q1_points:
    x, y = point

    if abs(x) == abs(y) and abs(x) > 1:
        point += rng.uniform(-SMALL_ERROR, SMALL_ERROR, size=2)

    if x == 0 and abs(y) > 1:
        point[0] += rng.uniform(-SMALL_ERROR, SMALL_ERROR)

    if y == 0 and abs(x) > 1:
        point[1] += rng.uniform(-SMALL_ERROR, SMALL_ERROR)

rng.shuffle(q1_points)

pseudo_angles = np.array(
    [pseudo_angle(point) for point in q1_points],
    dtype=np.float64,
)

# Reutiliza o Merge Sort C++ da Lista 1 para obter a ordem dos valores.
sorted_pseudo_angles = pseudo_angles.copy()
sort(sorted_pseudo_angles, method="merge", backend="cpp")

angular_order = np.searchsorted(sorted_pseudo_angles, pseudo_angles) + 1

In [4]:
# Cada painel é criado como uma figura simples e composto ao final.
input_order = np.arange(1, len(q1_points) + 1)

panel_data = [
    ("Ordem de entrada", input_order, [str(v) for v in input_order]),
    ("Pseudo-ângulo", pseudo_angles, [f"{v:.3f}" for v in pseudo_angles]),
    ("Ordem angular", angular_order, [str(v) for v in angular_order]),
]

q1_figures = []

for _, marker_values, labels in panel_data:
    panel = make_figure()

    panel.add_trace(
        go.Scatter(
            x=q1_points[:, 0],
            y=q1_points[:, 1],
            mode="markers+text",
            text=labels,
            textposition="top center",
            marker=dict(
                size=9,
                color=marker_values,
                colorscale="Turbo",
                showscale=False,
            ),
            customdata=pseudo_angles,
            hovertemplate=(
                "x=%{x:.6f}<br>"
                "y=%{y:.6f}<br>"
                "pseudo_angle=%{customdata:.6f}"
                "<extra></extra>"
            ),
            showlegend=False,
        )
    )

    q1_figures.append(panel)

fig = compose_figures(
    q1_figures,
    titles=[title for title, _, _ in panel_data],
    title="Questão 1 — Pseudo-ângulo e ordenação angular",
    rows=1,
    cols=3,
    width=1450,
    height=520,
    showlegend=False,
)

save_figure(fig, "q01_pseudo_angle.html").show()

Figura salva em: assignments\02-geometry\artifacts\q01_pseudo_angle.html


## Questão 2 — Função `entre(u, v, w)`

A função `entre(u, v, w)` utiliza o sinal do produto vetorial para determinar a posição angular de $w$ em relação ao ângulo convexo determinado por $u$ e $v$.

A convenção adotada é:

- `0`: $u$ e $v$ são colineares;
- `1`: $w$ pertence ao ângulo convexo, incluindo sua fronteira;
- `2`: $w$ está fora desse ângulo.

A decisão depende apenas de produtos vetoriais e comparações de sinais.

In [5]:
# Três orientações diferentes do ângulo convexo.
q2_cases = [
    (
        "Caso 1",
        np.array([0.5, 0.0], dtype=np.float64),
        np.array([0.0, 0.5], dtype=np.float64),
    ),
    (
        "Caso 2",
        np.array([0.0, -0.5], dtype=np.float64),
        np.array([-0.5, 0.0], dtype=np.float64),
    ),
    (
        "Caso 3",
        np.array([2.5, 1.5], dtype=np.float64),
        np.array([-1.5, -1.5], dtype=np.float64),
    ),
]

q2_points = q1_points.copy()

u = np.array([1.0, 0.0], dtype=np.float64)
v = np.array([0.0, 1.0], dtype=np.float64)

print("w dentro:      ", entre(u, v, np.array([1.0, 1.0], dtype=np.float64)))
print("w fora:        ", entre(u, v, np.array([-1.0, -1.0], dtype=np.float64)))
print("w = u:         ", entre(u, v, u))
print("w = v:         ", entre(u, v, v))

# Caso degenerado previsto no contrato.
u_collinear = np.array([1.0, 0.0], dtype=np.float64)
v_collinear = np.array([2.0, 0.0], dtype=np.float64)

print(
    "u,v colineares:",
    entre(u_collinear, v_collinear, np.array([0.0, 1.0], dtype=np.float64)),
)

w dentro:       1
w fora:         2
w = u:          1
w = v:          1
u,v colineares: 0


In [6]:
ENTRE_COLORS = {
    0: "gray",
    1: "seagreen",
    2: "crimson",
}

q2_figures = []

for _, u, v in q2_cases:
    panel = make_figure()

    classes = np.array(
        [entre(u, v, point) for point in q2_points],
        dtype=np.int64,
    )

    # Pontos classificados como colineares, dentro ou fora do ângulo.
    for category in (0, 1, 2):
        mask = classes == category

        panel.add_trace(
            go.Scatter(
                x=q2_points[mask, 0],
                y=q2_points[mask, 1],
                mode="markers",
                marker=dict(size=9, color=ENTRE_COLORS[category]),
                name=f"{category}",
                legendgroup=f"entre-{category}",
                customdata=classes[mask],
                hovertemplate=(
                    "x=%{x:.6f}<br>"
                    "y=%{y:.6f}<br>"
                    "entre=%{customdata}"
                    "<extra></extra>"
                ),
            )
        )

    # Vetores u e v que delimitam o ângulo analisado.
    for vector, label in ((u, "u"), (v, "v")):
        panel.add_trace(
            go.Scatter(
                x=[0.0, vector[0]],
                y=[0.0, vector[1]],
                mode="lines+markers+text",
                text=["", label],
                textposition="top center",
                line=dict(width=3),
                marker=dict(size=8),
                name=label,
                showlegend=False,
                hoverinfo="skip",
            )
        )

    q2_figures.append(panel)

fig = compose_figures(
    q2_figures,
    titles=[name for name, _, _ in q2_cases],
    title="Questão 2 — Classificação de w pelo ângulo convexo entre u e v",
    rows=1,
    cols=3,
    width=1400,
    height=520,
    showlegend=True,
)

fig.update_layout(legend_title="entre")
save_figure(fig, "q02_entre.html").show()

Figura salva em: assignments\02-geometry\artifacts\q02_entre.html


## Questão 3 — Localização de ponto em relação a um triângulo

Um ponto $P$ pode ser escrito em coordenadas baricêntricas como

$$
P=\lambda_1P_1+\lambda_2P_2+\lambda_3P_3,
\qquad
\lambda_1+\lambda_2+\lambda_3=1.
$$

Os sinais dessas coordenadas determinam a localização:

- todas positivas: ponto interior;
- nenhuma negativa e pelo menos uma nula: ponto na fronteira;
- pelo menos uma negativa: ponto exterior.

A implementação utiliza `oriented_area`, `barycentric_coordinates` e `locate_point_in_triangle`. A área orientada também indica a orientação dos vértices: positiva para ordem anti-horária e negativa para ordem horária.

In [7]:
triangle = np.array(
    [
        [-2.0, -1.0],
        [2.0, -1.0],
        [0.0, 2.0],
    ],
    dtype=np.float64,
)

# A inversão dos vértices deve apenas inverter o sinal da área.
print(f"Área orientada CCW: {oriented_area(triangle):.2f}")
print(f"Área orientada CW:  {oriented_area(triangle[::-1].copy()):.2f}")

# Grade regular usada para observar as regiões baricêntricas.
q3_values = [-3, -2, -1, 0, 1, 2, 3]
q3_points = np.array(
    [(x, y) for x in q3_values for y in q3_values],
    dtype=np.float64,
)

q3_barycentric = np.array(
    [barycentric_coordinates(point, triangle) for point in q3_points],
    dtype=np.float64,
)

q3_locations = np.array(
    [locate_point_in_triangle(point, triangle) for point in q3_points],
    dtype=np.int64,
)

examples = [
    np.array([0.0, 0.0], dtype=np.float64),
    triangle[0],
    triangle[1],
    triangle[2],
    (triangle[0] + 2.0 * triangle[2]) / 3.0,
]

for point in examples:
    bary = barycentric_coordinates(point, triangle)
    location = locate_point_in_triangle(point, triangle)

    print(
        f"P=({point[0]:.2f}, {point[1]:.2f})  "
        f"λ=({bary[0]:.3f}, {bary[1]:.3f}, {bary[2]:.3f})  "
        f"classe={location} ({LOCATION_LABELS[location]})"
    )

Área orientada CCW: 6.00
Área orientada CW:  -6.00
P=(0.00, 0.00)  λ=(0.333, 0.333, 0.333)  classe=1 (interior)
P=(-2.00, -1.00)  λ=(1.000, 0.000, 0.000)  classe=2 (fronteira)
P=(2.00, -1.00)  λ=(0.000, 1.000, 0.000)  classe=2 (fronteira)
P=(0.00, 2.00)  λ=(0.000, 0.000, 1.000)  classe=2 (fronteira)
P=(-0.67, 1.00)  λ=(0.333, 0.000, 0.667)  classe=2 (fronteira)


In [8]:
triangle_closed = close_path(triangle)
lambda_limit = np.max(np.abs(q3_barycentric))
q3_figures = []

# Três painéis mostram o valor de cada coordenada baricêntrica.
for index in range(3):
    panel = make_figure()

    lambda_values = q3_barycentric[:, index]

    panel.add_trace(
        go.Scatter(
            x=q3_points[:, 0],
            y=q3_points[:, 1],
            mode="markers+text",
            text=[f"{value:.2f}" for value in lambda_values],
            textposition="top center",
            textfont=dict(size=9),
            marker=dict(
                size=9,
                color=lambda_values,
                colorscale="RdBu",
                cmin=-lambda_limit,
                cmax=lambda_limit,
                showscale=False,
            ),
            customdata=lambda_values,
            hovertemplate=(
                "x=%{x:.3f}<br>"
                "y=%{y:.3f}<br>"
                f"λ{index + 1}=%{{customdata:.6f}}"
                "<extra></extra>"
            ),
            showlegend=False,
        )
    )

    add_polygon(
        panel,
        triangle,
        showlegend=False,
    )

    q3_figures.append(panel)

# O quarto painel mostra diretamente a classe retornada pelo kernel.
classification = make_figure()

for location in (0, 1, 2):
    mask = q3_locations == location

    classification.add_trace(
        go.Scatter(
            x=q3_points[mask, 0],
            y=q3_points[mask, 1],
            mode="markers",
            marker=dict(size=9, color=LOCATION_COLORS[location]),
            name=f"{location} — {LOCATION_LABELS[location]}",
            customdata=q3_locations[mask],
            hovertemplate=(
                "x=%{x:.3f}<br>"
                "y=%{y:.3f}<br>"
                "classe=%{customdata}"
                "<extra></extra>"
            ),
        )
    )

add_polygon(classification, triangle, showlegend=False)
q3_figures.append(classification)

fig = compose_figures(
    q3_figures,
    titles=["λ1", "λ2", "λ3", "Classificação"],
    title="Questão 3 — Coordenadas baricêntricas e localização",
    rows=2,
    cols=2,
    width=1050,
    height=900,
    showlegend=True,
)

fig.update_layout(legend_title="Classe")
save_figure(fig, "q03_barycentric_location.html").show()

Figura salva em: assignments\02-geometry\artifacts\q03_barycentric_location.html


## Questão 4 — Clipping de um segmento por um triângulo

Considere o segmento

$$
P(t)=tA+(1-t)B,
\qquad t\in[0,1],
$$

e sejam $(\alpha_1,\alpha_2,\alpha_3)$ e $(\beta_1,\beta_2,\beta_3)$ as coordenadas baricêntricas de $A$ e $B$, respectivamente.

Como a área orientada é afim em relação ao ponto variável,

$$
S_{P(t)P_2P_3}
=
tS_{AP_2P_3}
+
(1-t)S_{BP_2P_3}.
$$

Dividindo pela área orientada de $P_1P_2P_3$,

$$
\theta_1(t)
=
t\alpha_1+(1-t)\beta_1.
$$

O mesmo vale para as três coordenadas:

$$
\boxed{
\theta_i(t)=t\alpha_i+(1-t)\beta_i,
\qquad i=1,2,3.
}
$$

Portanto, as coordenadas baricêntricas variam linearmente ao longo de $AB$.

In [9]:
# Exemplo usado para verificar as três partes da questão.
triangle_q4 = np.array(
    [
        [0.0, 0.0],
        [4.0, 0.0],
        [0.0, 4.0],
    ],
    dtype=np.float64,
)

A = np.array([-1.0, 1.0], dtype=np.float64)
B = np.array([3.0, 3.0], dtype=np.float64)

alpha = np.array(
    barycentric_coordinates(A, triangle_q4),
    dtype=np.float64,
)

beta = np.array(
    barycentric_coordinates(B, triangle_q4),
    dtype=np.float64,
)

print(f"α(A) = {alpha}")
print(f"β(B) = {beta}")

α(A) = [ 1.   -0.25  0.25]
β(B) = [-0.5   0.75  0.75]


### a) Segmento completamente contido

O triângulo é convexo. Logo, $AB$ está completamente contido, incluindo a fronteira, se e somente se os dois extremos pertencem ao triângulo:

$$
\boxed{
\alpha_i\geq0
\;\land\;
\beta_i\geq0,
\qquad i=1,2,3.
}
$$

De fato, cada $\theta_i(t)$ é uma combinação convexa de $\alpha_i$ e $\beta_i$.

### b) Condição suficiente para segmento completamente exterior

Uma condição suficiente para garantir que todo o segmento permaneça exterior é existir uma mesma coordenada negativa nos dois extremos:

$$
\boxed{
\exists i:
\alpha_i<0
\;\land\;
\beta_i<0.
}
$$

Nesse caso, $\theta_i(t)<0$ para todo $t\in[0,1]$, ou seja, todo o segmento permanece no mesmo semiplano exterior determinado pelo lado oposto a $P_i$.

Essa condição é suficiente, mas não necessária.

In [10]:
# Verifica diretamente as condições dos itens (a) e (b) para o exemplo.
contained = bool(np.all(alpha >= 0.0) and np.all(beta >= 0.0))
same_exterior_coordinate = bool(np.any((alpha < 0.0) & (beta < 0.0)))

print(f"AB completamente contido: {contained}")
print(
    "Teste suficiente de exterioridade: "
    f"{same_exterior_coordinate}"
)

# Neste exemplo, A e B são exteriores, mas por coordenadas negativas diferentes.
print(f"A: {LOCATION_LABELS[locate_point_in_triangle(A, triangle_q4)]}")
print(f"B: {LOCATION_LABELS[locate_point_in_triangle(B, triangle_q4)]}")

AB completamente contido: False
Teste suficiente de exterioridade: False
A: exterior
B: exterior


### c) Interseções com as retas dos lados

O lado oposto a $P_i$ é caracterizado por

$$
\theta_i(t)=0.
$$

Assim,

$$
t\alpha_i+(1-t)\beta_i=0,
$$

e, quando $\alpha_i\neq\beta_i$,

$$
\boxed{
t_i=\frac{\beta_i}{\beta_i-\alpha_i}
}
$$

com ponto correspondente

$$
\boxed{
X_i=t_iA+(1-t_i)B.
}
$$

Para que a interseção pertença ao próprio segmento $AB$, é necessário $0\leq t_i\leq1$. Para que $X_i$ pertença ao lado do triângulo, e não apenas à reta que o contém, as outras duas coordenadas baricêntricas também devem ser não negativas.

Se $\alpha_i=\beta_i\neq0$, $AB$ é paralelo à reta correspondente. Se $\alpha_i=\beta_i=0$, todo o segmento pertence a essa reta.

In [11]:
q4_intersections = []

# Calcula as interseções com as três retas associadas a θ_i = 0.
for i, (alpha_i, beta_i) in enumerate(zip(alpha, beta), start=1):
    if np.isclose(alpha_i, beta_i):
        print(f"θ{i}: sem interseção isolada")
        continue

    t_i = beta_i / (beta_i - alpha_i)
    x_i = t_i * A + (1.0 - t_i) * B
    q4_intersections.append((i, t_i, x_i))

    print(
        f"θ{i}=0 -> t={t_i:.6f}, "
        f"X=({x_i[0]:.6f}, {x_i[1]:.6f}), "
        f"no segmento={0.0 <= t_i <= 1.0}"
    )

# Pontos representativos confirmam a mudança de classe ao longo de AB.
t_test = np.array([0.0, 1.0 / 3.0, 0.5, 3.0 / 4.0, 1.0])

for t_i in t_test:
    point = t_i * A + (1.0 - t_i) * B
    location = locate_point_in_triangle(point, triangle_q4)

    print(
        f"t={t_i:.3f}  "
        f"P=({point[0]:.3f}, {point[1]:.3f})  "
        f"classe={location} ({LOCATION_LABELS[location]})"
    )

θ1=0 -> t=0.333333, X=(1.666667, 2.333333), no segmento=True
θ2=0 -> t=0.750000, X=(0.000000, 1.500000), no segmento=True
θ3=0 -> t=1.500000, X=(-3.000000, 0.000000), no segmento=False
t=0.000  P=(3.000, 3.000)  classe=0 (exterior)
t=0.333  P=(1.667, 2.333)  classe=2 (fronteira)
t=0.500  P=(1.000, 2.000)  classe=1 (interior)
t=0.750  P=(0.000, 1.500)  classe=2 (fronteira)
t=1.000  P=(-1.000, 1.000)  classe=0 (exterior)


### Resultado do exemplo

Para

$$
A=(-1,1),
\qquad
B=(3,3),
$$

temos

$$
\alpha=\left(1,-\frac14,\frac14\right),
\qquad
\beta=\left(-\frac12,\frac34,\frac34\right).
$$

Os dois extremos são exteriores, mas o teste suficiente do item (b) não descarta o segmento, pois não existe uma mesma coordenada negativa em ambos.

As interseções relevantes são

$$
t_1=\frac13,
\qquad
X_1=\left(\frac53,\frac73\right),
$$

e

$$
t_2=\frac34,
\qquad
X_2=\left(0,\frac32\right).
$$

A terceira solução fornece $t_3=\frac32$, fora de $[0,1]$. Portanto, a porção de $AB$ contida no triângulo é

$$
\boxed{
\frac13\leq t\leq\frac34.
}
$$

In [12]:
# Painel 1: evolução das três coordenadas baricêntricas.
barycentric_fig = make_figure(
    x_title="t",
    y_title="θᵢ(t)",
    equal_axes=False,
)

t_plot = np.linspace(0.0, 1.0, 201)
theta_plot = (
    t_plot[:, None] * alpha[None, :]
    + (1.0 - t_plot[:, None]) * beta[None, :]
)

for i in range(3):
    barycentric_fig.add_trace(
        go.Scatter(
            x=t_plot,
            y=theta_plot[:, i],
            mode="lines",
            name=f"θ{i + 1}",
        )
    )

barycentric_fig.add_trace(
    go.Scatter(
        x=[0.0, 1.0],
        y=[0.0, 0.0],
        mode="lines",
        line=dict(dash="dash"),
        name="θ = 0",
        showlegend=False,
        hoverinfo="skip",
    )
)

# Painel 2: classificação geométrica de pontos ao longo do segmento.
geometry_fig = make_figure()
add_polygon(geometry_fig, triangle_q4, name="Triângulo")

geometry_fig.add_trace(
    go.Scatter(
        x=[A[0], B[0]],
        y=[A[1], B[1]],
        mode="lines",
        name="Segmento AB",
        line=dict(width=2, dash="dash"),
        hoverinfo="skip",
    )
)

valid_intersections = [
    t_i
    for _, t_i, _ in q4_intersections
    if 0.0 <= t_i <= 1.0
]

t_values = np.unique(
    np.concatenate(
        (
            np.linspace(0.0, 1.0, 11),
            np.asarray(valid_intersections, dtype=np.float64),
        )
    )
)

segment_points = np.array(
    [t_i * A + (1.0 - t_i) * B for t_i in t_values],
    dtype=np.float64,
)

locations = np.array(
    [locate_point_in_triangle(point, triangle_q4) for point in segment_points],
    dtype=np.int64,
)

for location in (0, 1, 2):
    mask = locations == location

    geometry_fig.add_trace(
        go.Scatter(
            x=segment_points[mask, 0],
            y=segment_points[mask, 1],
            mode="markers",
            marker=dict(size=10, color=LOCATION_COLORS[location]),
            name=f"{location} — {LOCATION_LABELS[location]}",
            customdata=t_values[mask],
            hovertemplate=(
                "t=%{customdata:.6f}<br>"
                "x=%{x:.6f}<br>"
                "y=%{y:.6f}"
                "<extra></extra>"
            ),
        )
    )

geometry_fig.add_trace(
    go.Scatter(
        x=[A[0], B[0]],
        y=[A[1], B[1]],
        mode="text",
        text=["A (t=1)", "B (t=0)"],
        textposition="bottom center",
        showlegend=False,
        hoverinfo="skip",
    )
)

fig = compose_figures(
    [barycentric_fig, geometry_fig],
    titles=[
        "Coordenadas baricêntricas ao longo de AB",
        "Classificação ao longo do segmento",
    ],
    title="Questão 4 — Clipping por coordenadas baricêntricas",
    rows=1,
    cols=2,
    width=1250,
    height=620,
    equal_axes=[False, True],
    showlegend=True,
)

# As anotações são adicionadas depois da composição para usar o eixo do 2º painel.
for i, t_i, x_i in q4_intersections:
    if 0.0 <= t_i <= 1.0:
        fig.add_annotation(
            x=x_i[0],
            y=x_i[1],
            text=f"X{i}<br>t={t_i:.3f}",
            showarrow=True,
            ax=35,
            ay=-45,
            xref="x2",
            yref="y2",
        )

fig.update_layout(legend_title="Resultados")
save_figure(fig, "q04_clipping.html").show()

Figura salva em: assignments\02-geometry\artifacts\q04_clipping.html


## Questão 5 — Triângulos disjuntos

Dois triângulos são considerados disjuntos quando não possuem qualquer ponto em comum. Assim, contato em um vértice ou em uma aresta também significa que eles **não** são disjuntos.

O algoritmo combina:

1. interseção entre pares de arestas;
2. contenção de um triângulo no outro.

Se nenhuma dessas situações ocorrer, os triângulos são disjuntos. Triângulos degenerados são rejeitados pelo contrato da implementação.

In [13]:
# T0 é comparado com seis configurações representativas.
triangles = {
    "T0_base": np.array(
        [[0.0, 0.0], [4.0, 0.0], [0.0, 4.0]],
        dtype=np.float64,
    ),
    "T1_disjoint": np.array(
        [[5.0, 5.0], [7.0, 5.0], [5.0, 7.0]],
        dtype=np.float64,
    ),
    "T2_overlap": np.array(
        [[2.0, 1.0], [5.0, 1.0], [2.0, 4.0]],
        dtype=np.float64,
    ),
    "T3_contained": np.array(
        [[0.5, 0.5], [1.5, 0.5], [0.5, 1.5]],
        dtype=np.float64,
    ),
    "T4_touch_vertex": np.array(
        [[4.0, 0.0], [6.0, 0.0], [4.0, 2.0]],
        dtype=np.float64,
    ),
    "T5_touch_edge": np.array(
        [[0.0, 0.0], [4.0, 0.0], [2.0, -2.0]],
        dtype=np.float64,
    ),
    "T6_crossing": np.array(
        [[2.0, -1.0], [5.0, 2.0], [2.0, 5.0]],
        dtype=np.float64,
    ),
}

triangle_descriptions = {
    "T0_base": "triângulo-base",
    "T1_disjoint": "completamente separado",
    "T2_overlap": "sobreposição parcial",
    "T3_contained": "completamente contido em T0",
    "T4_touch_vertex": "toca T0 em um vértice",
    "T5_touch_edge": "compartilha uma aresta com T0",
    "T6_crossing": "cruza T0",
}

base_name = "T0_base"
base_triangle = triangles[base_name]

for name, triangle in triangles.items():
    result = triangles_are_disjoint(base_triangle, triangle)
    print(
        f"{base_name} vs {name:16s} -> "
        f"{result} ({triangle_descriptions[name]})"
    )

# Matriz completa para verificar simetria e todos os pares disponíveis.
names = list(triangles)
disjoint_matrix = pd.DataFrame(index=names, columns=names, dtype=bool)

for row_name in names:
    for col_name in names:
        disjoint_matrix.loc[row_name, col_name] = triangles_are_disjoint(
            triangles[row_name],
            triangles[col_name],
        )

disjoint_matrix

T0_base vs T0_base          -> False (triângulo-base)
T0_base vs T1_disjoint      -> True (completamente separado)
T0_base vs T2_overlap       -> False (sobreposição parcial)
T0_base vs T3_contained     -> False (completamente contido em T0)
T0_base vs T4_touch_vertex  -> False (toca T0 em um vértice)
T0_base vs T5_touch_edge    -> False (compartilha uma aresta com T0)
T0_base vs T6_crossing      -> False (cruza T0)


,T0_base,T1_disjoint,T2_overlap,T3_contained,T4_touch_vertex,T5_touch_edge,T6_crossing
T0_base,False,True,False,False,False,False,False
T1_disjoint,True,False,True,True,True,True,True
T2_overlap,False,True,False,True,False,True,False
T3_contained,False,True,True,False,True,True,True
T4_touch_vertex,False,True,False,True,False,False,False
T5_touch_edge,False,True,True,True,False,False,False
T6_crossing,False,True,False,True,False,False,False


In [14]:
comparison_names = [
    name
    for name in triangles
    if name != base_name
]

q5_figures = []
q5_titles = []

for name in comparison_names:
    result = triangles_are_disjoint(base_triangle, triangles[name])
    panel = make_figure()

    # O triângulo-base é mantido em todos os painéis para comparação.
    add_polygon(
        panel,
        base_triangle,
        name="T0_base",
        fill=True,
        fillcolor="rgba(0, 100, 255, 0.20)",
        showlegend=False,
    )
    add_polygon(
        panel,
        triangles[name],
        name=name,
        fill=True,
        fillcolor="rgba(255, 100, 0, 0.20)",
        showlegend=False,
    )

    q5_figures.append(panel)
    q5_titles.append(
        f"{name}<br>{triangle_descriptions[name]}<br>disjoint = {result}"
    )

fig = compose_figures(
    q5_figures,
    titles=q5_titles,
    title="Questão 5 — Casos de disjunção entre triângulos",
    rows=2,
    cols=3,
    width=1200,
    height=850,
    vertical_spacing=0.16,
    showlegend=False,
)

fig.update_layout(
    title=dict(
        x=0.5,
        xanchor="center",
        y=0.98,
        yanchor="top",
    ),
    margin=dict(t=100),
)

for annotation in fig.layout.annotations:
    annotation.update(yshift=-4)

save_figure(fig, "q05_triangle_disjointness.html").show()

Figura salva em: assignments\02-geometry\artifacts\q05_triangle_disjointness.html


## Questão 6 — Convexidade de um polígono

Para cada três vértices consecutivos $P_i$, $P_{i+1}$ e $P_{i+2}$, calcula-se o sinal

$$
(P_{i+1}-P_i)\times(P_{i+2}-P_{i+1}).
$$

Assumindo que a sequência define um polígono simples, ele é convexo quando todos esses giros possuem o mesmo sinal. A orientação global não importa: todos podem ser positivos (anti-horário) ou negativos (horário).

Nesta implementação, três vértices consecutivos colineares retornam `False`, pois foi adotada uma representação mínima, sem vértices intermediários redundantes.

Como cada vértice é analisado uma única vez,

$$
\boxed{T(n)=O(n)}.
$$

In [15]:
convex = np.array(
    [[0.0, 0.0], [3.0, 0.0], [4.0, 2.0], [2.0, 4.0], [0.0, 2.0]],
    dtype=np.float64,
)

concave = np.array(
    [[0.0, 0.0], [3.0, 0.0], [1.5, 1.0], [3.0, 3.0], [0.0, 3.0]],
    dtype=np.float64,
)

collinear_vertex = np.array(
    [[0.0, 0.0], [2.0, 0.0], [4.0, 0.0], [4.0, 3.0], [0.0, 3.0]],
    dtype=np.float64,
)

# Mesmo polígono convexo, agora percorrido no sentido horário.
convex_clockwise = convex[::-1].copy()

q6_polygons = {
    "Convex CCW": convex,
    "Convex CW": convex_clockwise,
    "Concave": concave,
    "Collinear vertex": collinear_vertex,
}

for name, polygon in q6_polygons.items():
    print(f"{name:18s}: {is_convex(polygon)}")

Convex CCW        : True
Convex CW         : True
Concave           : False
Collinear vertex  : False


In [16]:
q6_figures = []
q6_titles = []

for name, polygon in q6_polygons.items():
    result = is_convex(polygon)
    color = "seagreen" if result else "crimson"

    panel = make_figure()

    add_polygon(
        panel,
        polygon,
        name=name,
        fill=True,
        vertex_labels=True,
        line_color=color,
        showlegend=False,
    )

    q6_figures.append(panel)
    q6_titles.append(f"{name}<br>is_convex = {result}")

fig = compose_figures(
    q6_figures,
    titles=q6_titles,
    title="Questão 6 — Verificação de convexidade",
    rows=2,
    cols=2,
    width=950,
    height=900,
    vertical_spacing=0.16,
    showlegend=False,
)

fig.update_layout(
    title=dict(
        x=0.5,
        xanchor="center",
        y=0.98,
        yanchor="top",
    ),
    margin=dict(t=130),
)

for annotation in fig.layout.annotations:
    annotation.update(yshift=10)

save_figure(fig, "q06_convexity.html").show()

Figura salva em: assignments\02-geometry\artifacts\q06_convexity.html


## Questão 7 — Localização a partir da primeira interseção

Considere um polígono simples

$$
P=p_1p_2\ldots p_n
$$

orientado no sentido horário, e um ponto $p_0$ cuja posição em relação ao polígono deseja-se determinar.

A partir de $p_0$, considere uma semirreta horizontal para a direita e seja $u$ seu primeiro ponto de interseção com a fronteira, ocorrendo no lado orientado $p_ip_{i+1}$.

Como $u$ é a **primeira interseção**, não existe fronteira entre $p_0$ e $u$. Logo, $p_0$ está na mesma região — interior ou exterior — que o trecho imediatamente anterior a $u$.

Como o polígono está orientado no sentido horário, seu interior encontra-se à **direita** de cada lado orientado. Definindo

$$
e=p_{i+1}-p_i,
$$

temos

$$
\boxed{
e\times(p_0-u)<0
\Rightarrow
p_0\text{ é interior},
}
$$

e

$$
\boxed{
e\times(p_0-u)>0
\Rightarrow
p_0\text{ é exterior}.
}
$$

Se $u=p_0$, o ponto está na fronteira. Se a semirreta não possuir interseção, $p_0$ é exterior. Portanto, depois de obter apenas $u$ e o lado associado, não é necessário armazenar ou contar as demais interseções.

In [17]:
# Polígono simples, côncavo e orientado no sentido horário.
polygon_q7 = np.array(
    [
        [0.0, 0.0],
        [-1.0, 4.0],
        [2.0, 1.5],
        [5.0, 3.5],
        [4.0, 0.0],
    ],
    dtype=np.float64,
)

q7_points = {
    "p_inside": np.array([1.0, 1.0], dtype=np.float64),
    "p_outside_left": np.array([-1.0, 1.5], dtype=np.float64),
    "p_outside_right": np.array([5.0, 2.0], dtype=np.float64),
    "p_outside_concavity": np.array([2.0, 2.5], dtype=np.float64),
    "p_boundary": np.array([2.0, 0.0], dtype=np.float64),
    "p_outside_to_vertex": np.array([-1.0, 0.0], dtype=np.float64),
}

print(
    f"Área orientada do polígono: "
    f"{oriented_area(polygon_q7):.3f} (horário < 0)"
)
print()

q7_results = {}

# Primeiro obtém apenas u e o lado; depois usa exatamente essa informação
# para classificar p0, como proposto no enunciado.
for name, point in q7_points.items():
    intersection = first_horizontal_intersection(point, polygon_q7)
    location = locate_from_first_intersection(
        point,
        polygon_q7,
        intersection,
    )

    q7_results[name] = (intersection, location)

    print(f"{name}:")
    print(f"  classificação = {LOCATION_LABELS[location]}")

    if intersection is None:
        print("  u = None")
    else:
        print(f"  u = {intersection.point}")
        print(f"  edge_index = {intersection.edge_index}")

    print()

Área orientada do polígono: -12.000 (horário < 0)

p_inside:
  classificação = interior
  u = [4.285714285714286, 1.0]
  edge_index = 3

p_outside_left:
  classificação = exterior
  u = [-0.375, 1.5]
  edge_index = 0

p_outside_right:
  classificação = exterior
  u = None

p_outside_concavity:
  classificação = exterior
  u = [3.5, 2.5]
  edge_index = 2

p_boundary:
  classificação = fronteira
  u = [2.0, 0.0]
  edge_index = 4

p_outside_to_vertex:
  classificação = exterior
  u = [0.0, 0.0]
  edge_index = 0



In [18]:
# A Q7 usa a figura padrão, mas mantém anotações próprias para p0, u e o lado.
fig = make_figure()
add_polygon(
    fig,
    polygon_q7,
    name="Polígono",
    fill=True,
    vertex_labels=True,
    line_color="black",
)

ray_end = max(
    polygon_q7[:, 0].max(),
    *(point[0] for point in q7_points.values()),
) + 1.0

# Deslocamentos apenas visuais para evitar sobreposição das anotações.
point_offsets = {
    "p_inside": (0, -75),
    "p_outside_left": (-30, -75),
    "p_outside_right": (0, -75),
    "p_outside_concavity": (0, -75),
    "p_boundary": (0, 70),
    "p_outside_to_vertex": (-35, 70),
}

for point_name, point in q7_points.items():
    intersection, location = q7_results[point_name]

    location_label = LOCATION_LABELS[location]
    color = LOCATION_COLORS[location]
    ax, ay = point_offsets[point_name]

    # Resultado de classificação produzido pelo kernel C++.
    fig.add_trace(
        go.Scatter(
            x=[point[0]],
            y=[point[1]],
            mode="markers",
            marker=dict(size=12, color=color),
            name=f"{point_name}: {location_label}",
            hovertemplate=(
                f"{point_name}<br>"
                f"resultado={location_label}<br>"
                "x=%{x:.3f}<br>"
                "y=%{y:.3f}"
                "<extra></extra>"
            ),
        )
    )

    fig.add_annotation(
        x=point[0],
        y=point[1],
        text=f"{point_name}<br>resultado: {location_label}",
        showarrow=True,
        arrowhead=2,
        ax=ax,
        ay=ay,
    )

    # Pontos de fronteira já têm u = p0.
    if location == 2:
        continue

    # Sem interseção, a semirreta segue livremente para a direita.
    if intersection is None:
        fig.add_trace(
            go.Scatter(
                x=[point[0], ray_end],
                y=[point[1], point[1]],
                mode="lines",
                line=dict(color=color, dash="dash"),
                showlegend=False,
                hoverinfo="skip",
            )
        )
        continue

    u = np.asarray(intersection.point, dtype=np.float64)
    edge_index = intersection.edge_index

    # Trecho entre p0 e a primeira interseção u.
    fig.add_trace(
        go.Scatter(
            x=[point[0], u[0]],
            y=[point[1], u[1]],
            mode="lines",
            line=dict(color=color, dash="dash"),
            showlegend=False,
            hoverinfo="skip",
        )
    )

    fig.add_trace(
        go.Scatter(
            x=[u[0]],
            y=[u[1]],
            mode="markers",
            marker=dict(size=10, color=color, symbol="x"),
            showlegend=False,
            hovertemplate=(
                f"u=({u[0]:.3f}, {u[1]:.3f})<br>"
                f"lado=P{edge_index + 1}→"
                f"P{(edge_index + 1) % len(polygon_q7) + 1}"
                "<extra></extra>"
            ),
        )
    )

    fig.add_annotation(
        x=u[0],
        y=u[1],
        text=(
            f"u = ({u[0]:.2f}, {u[1]:.2f})"
            f"<br>P{edge_index + 1} → "
            f"P{(edge_index + 1) % len(polygon_q7) + 1}"
        ),
        showarrow=True,
        arrowhead=2,
        ax=60,
        ay=45,
    )

fig.update_layout(
    title="Questão 7 — Localização pela primeira interseção",
    width=1050,
    height=780,
    legend_title="Classificação",
)

save_figure(fig, "q07_first_intersection.html").show()

Figura salva em: assignments\02-geometry\artifacts\q07_first_intersection.html


## Artefatos gerados

Ao executar o notebook, as visualizações são regeneradas em:

```text
assignments/02-geometry/artifacts/
├── q01_pseudo_angle.html
├── q02_entre.html
├── q03_barycentric_location.html
├── q04_clipping.html
├── q05_triangle_disjointness.html
├── q06_convexity.html
└── q07_first_intersection.html
```

Os arquivos HTML preservam zoom, hover e demais recursos interativos do Plotly.